# Netflix Content Strategy Analysis

This notebook explores Netflix's library to study content mix, genre trends, country production patterns, and growth over time.

Place the dataset at `data/netflix_titles.csv` before running the analysis.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import seaborn as sns

NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / 'src').exists():
    PROJECT_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / 'src').exists():
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analysis import (
    get_country_trend,
    get_growth_trends,
    get_genre_trend,
    get_top_countries,
    get_top_genres,
    get_type_distribution,
    get_type_distribution_by_year,
    summarize_content_focus,
)
from src.cleaning import prepare_netflix_dataset
from src.visualization import (
    plot_heatmap,
    plot_horizontal_bar,
    plot_line,
    plot_pie,
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 7)

CHART_DIR = PROJECT_ROOT / 'outputs' / 'charts'
CHART_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT

WindowsPath('C:/Users/Melwin/Work/Netflix-Content-Strategy-Analysis')

## 0. Download Dataset

This step uses KaggleHub to download the latest version of the Netflix dataset before the analysis runs.

In [4]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('kagglehub') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'kagglehub'])

kagglehub = importlib.import_module('kagglehub')

# Download the latest dataset snapshot from Kaggle
kaggle_path = Path(kagglehub.dataset_download('shivamb/netflix-shows'))
csv_candidates = sorted(kaggle_path.rglob('*.csv'))
if not csv_candidates:
    raise FileNotFoundError(f'No CSV files were found in {kaggle_path}.')

DATA_PATH = csv_candidates[0]
print('Path to dataset files:', kaggle_path)
print('Using dataset CSV:', DATA_PATH)

C:\Users\Melwin\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 1.34M/1.34M [00:01<00:00, 871kB/s]

Extracting files...
Path to dataset files: C:\Users\Melwin\.cache\kagglehub\datasets\shivamb\netflix-shows\versions\5
Using dataset CSV: C:\Users\Melwin\.cache\kagglehub\datasets\shivamb\netflix-shows\versions\5\netflix_titles.csv


In [5]:
df = prepare_netflix_dataset(DATA_PATH)

print(f'Dataset shape: {df.shape}')
print('Columns:')
print(df.columns.tolist())

df.head()

Dataset shape: (8807, 14)
Columns:
['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description', 'added_year', 'added_month']


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,added_year,added_month
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,<NA>,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",2021.0,2021-09
1,s2,TV Show,Blood & Water,<NA>,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2021.0,2021-09
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",<NA>,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,2021.0,2021-09
3,s4,TV Show,Jailbirds New Orleans,<NA>,<NA>,<NA>,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",2021.0,2021-09
4,s5,TV Show,Kota Factory,<NA>,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2021.0,2021-09


## 1. Movies vs TV Shows Distribution

This section checks whether Netflix's catalog is weighted more toward films or series and how that mix varies by release year.

In [ ]:
type_distribution = get_type_distribution(df)
type_yearly = get_type_distribution_by_year(df)
type_yearly_pivot = type_yearly.pivot(index='release_year', columns='type', values='count').fillna(0)

type_distribution

In [ ]:
plot_pie(type_distribution.set_index('type')['count'], 'Netflix Movies vs TV Shows', CHART_DIR / '01_type_distribution_pie.png')
plot_horizontal_bar(type_distribution, x='count', y='type', title='Netflix Movies vs TV Shows')
plot_line(type_yearly_pivot, title='Movies vs TV Shows by Release Year', ylabel='Titles', output_path=CHART_DIR / '01_type_trend.png')
plt.show()

## 2. Genre Popularity Over Time

Netflix titles often include multiple genres, so this section explodes the genre list before measuring frequency and release-year trends.

In [ ]:
top_genres = get_top_genres(df, top_n=10)
genre_trend = get_genre_trend(df, top_n=8)

top_genres

In [ ]:
plot_horizontal_bar(top_genres, x='count', y='genre', title='Top 10 Netflix Genres', output_path=CHART_DIR / '02_top_genres.png')
plot_line(genre_trend, title='Top Genre Trends by Release Year', ylabel='Titles', output_path=CHART_DIR / '02_genre_trends.png')
plot_heatmap(genre_trend.tail(20).T, title='Genre Concentration in Recent Release Years', output_path=CHART_DIR / '02_genre_heatmap.png')
plt.show()

## 3. Country-wise Content Production

This section ranks the strongest production markets and shows how international catalog growth has developed across release years.

In [ ]:
top_countries = get_top_countries(df, top_n=10)
country_trend = get_country_trend(df, top_n=8)

top_countries

In [ ]:
plot_horizontal_bar(top_countries, x='count', y='country', title='Top Producing Countries', output_path=CHART_DIR / '03_top_countries.png')
plot_line(country_trend, title='Top Country Trends by Release Year', ylabel='Titles', output_path=CHART_DIR / '03_country_trends.png')
plt.show()

## 4. Content Growth Trends

The final section measures how quickly Netflix expanded its catalog by year and month using the title addition date.

In [ ]:
yearly_growth, monthly_growth = get_growth_trends(df)
summary = summarize_content_focus(df)

summary

In [ ]:
plot_line(yearly_growth, title='Titles Added Per Year', ylabel='Titles Added', output_path=CHART_DIR / '04_yearly_growth.png')
plot_line(monthly_growth.tail(36), title='Recent Monthly Title Additions', ylabel='Titles Added', output_path=CHART_DIR / '04_monthly_growth.png')
plt.show()

## Key Business Takeaways

- Netflix's catalog mix can be compared directly through the type distribution and yearly trend views.
- Genre and country explosions reveal how much of the platform depends on international and multi-genre titles.
- Title-addition trends show the point at which catalog expansion accelerated most sharply.

Exported charts are saved in `outputs/charts/`.